# ArmGPT - Build Your Own Armenian Language Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EdikSimonian/armenian-gpt/blob/main/notebooks/armgpt_colab.ipynb)

In this notebook, you will:
1. Download Armenian text data (from Wikipedia)
2. Build a vocabulary (mapping characters to numbers)
3. Train a GPT language model
4. Generate Armenian text!
5. **(Stage 2)** Fine-tune it to answer questions like a chatbot

**No prior experience needed** - just run each cell from top to bottom.

**Important:** Go to **Runtime > Change runtime type > T4 GPU** before starting!

In [ ]:
#@title Step 0: Setup (run this first!)
import os

# Always start from /content to avoid path issues
os.chdir('/content')

# Clone fresh or update existing repo
if os.path.exists('armenian-gpt'):
    print('Repo exists, pulling latest changes...')
    os.chdir('armenian-gpt')
    !git pull
else:
    print('Cloning repository...')
    !git clone https://github.com/EdikSimonian/armenian-gpt.git
    os.chdir('armenian-gpt')

# Install dependencies
!pip install -q torch numpy requests datasets

# Verify we're in the right place
print(f'\nWorking directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

# Check GPU
import torch
if torch.cuda.is_available():
    print(f'\nGPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB')
else:
    print('\nNo GPU found. Go to Runtime > Change runtime type > T4 GPU')

---
## Stage 1: Train a Base Language Model

### Step 1: Download Armenian Text Data

We'll download text from Armenian Wikipedia (~500 MB). This takes ~5 minutes.

In [ ]:
!python data/download.py

### Step 2: Prepare the Data

Clean the text, build a vocabulary, convert to numbers, split into train/val.

In [ ]:
!python data/prepare.py --tokenizer char

### Step 2b: Explore the Data

In [ ]:
import json
import numpy as np

# Load vocabulary
with open('data/tokenizer.json', 'r', encoding='utf-8') as f:
    tok_data = json.load(f)
vocab = tok_data['itos']
print(f'Vocabulary size: {len(vocab)} characters\n')

# Show all characters
for i, ch in enumerate(vocab):
    display = repr(ch) if ch in (' ', '\n', '\t') else ch
    print(f'  {i:3d} -> {display}', end='   ')
    if (i + 1) % 6 == 0:
        print()

# Load and show training data stats
train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')
print(f'\n\nTraining tokens:   {len(train_data):,}')
print(f'Validation tokens: {len(val_data):,}')

# Show a sample
sample = ''.join(vocab[i] for i in train_data[:300] if i < len(vocab))
print(f'\nSample text:\n{sample}')

In [ ]:
# Visualize character frequency
import matplotlib.pyplot as plt

counts = np.bincount(train_data, minlength=len(vocab))
top_n = 40
top_idx = np.argsort(counts)[-top_n:][::-1]
labels = [repr(vocab[i]) if vocab[i] in (' ', '\n') else vocab[i] for i in top_idx if i < len(vocab)]
values = [counts[i] for i in top_idx if i < len(vocab)]

plt.figure(figsize=(14, 4))
plt.bar(range(len(labels)), values)
plt.xticks(range(len(labels)), labels, fontsize=9)
plt.ylabel('Frequency')
plt.title('Most Common Characters in Armenian Training Data')
plt.tight_layout()
plt.show()

### Step 3: Train the Model

Watch the **loss** go down - that means the model is learning!

| Preset | Time | Hardware |
|--------|------|----------|
| `tiny` | ~1 min | CPU |
| `small` | ~30 min | GPU (T4) |
| `medium` | ~2 hours | Good GPU |

In [ ]:
# Change this to 'tiny' for a quick test, or 'medium' for best results
PRESET = 'small'

!python train.py --preset $PRESET

In [ ]:
# Plot training metrics
import json
import matplotlib.pyplot as plt

with open('checkpoints/metrics.json', 'r') as f:
    m = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(m['steps'], m['train_loss'], 'o-', label='Train')
axes[0].plot(m['steps'], m['val_loss'], 's-', label='Validation')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (lower = better)'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(m['steps'], m['perplexity'], 'o-', color='green')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity (lower = better)'); axes[1].grid(alpha=0.3)

axes[2].plot(m['steps'], m['accuracy'], 'o-', color='orange')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Accuracy (higher = better)'); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Final: loss={m['val_loss'][-1]:.3f}, perplexity={m['perplexity'][-1]:.1f}, accuracy={m['accuracy'][-1]:.1f}%")

### Step 4: Generate Armenian Text!

**Temperature** controls randomness: 0.3 = safe, 0.8 = balanced, 1.5 = wild

In [ ]:
PROMPT = 'Հայաստանի'      # <-- change this!
TEMPERATURE = 0.8           # <-- try 0.3, 0.8, or 1.5

!python generate.py --prompt "$PROMPT" --temperature $TEMPERATURE --length 400

In [ ]:
# Compare temperatures
for temp in [0.3, 0.8, 1.5]:
    print(f"{'='*50}")
    print(f"Temperature = {temp}")
    print(f"{'='*50}")
    !python generate.py --prompt "Երևանը" --temperature $temp --length 150
    print()

---
## Stage 2: Make It Conversational

Stage 1 generates text like autocomplete. Now we'll fine-tune it on 52K Armenian Q&A pairs so it can **answer questions** like a chatbot.

This is exactly how ChatGPT is made:
1. Stage 1: Train on text → learns language
2. Stage 2: Fine-tune on conversations → learns to answer questions

In [ ]:
# Download and prepare 52K Armenian Q&A pairs (Alpaca-Armenian)
!python data/prepare_chat.py

In [ ]:
# Fine-tune on conversations (uses your Stage 1 model as starting point)
!python finetune.py

In [ ]:
# Test the chatbot! Try different questions.
import torch
import json
import sys
sys.path.insert(0, '.')
from model import GPT
from tokenizers.char_tokenizer import CharTokenizer

# Load model
ckpt = torch.load('checkpoints/chat_final.pt', map_location='cuda' if torch.cuda.is_available() else 'cpu', weights_only=False)
cfg = ckpt['config']
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = CharTokenizer.load('data_chat/tokenizer.json')
model = GPT(
    vocab_size=tokenizer.vocab_size,
    n_layer=cfg['n_layer'], n_head=cfg['n_head'], n_embd=cfg['n_embd'],
    block_size=cfg['block_size'], dropout=0.0
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()

end_id = tokenizer.stoi.get('<|end|>')
stop = {end_id} if end_id else None

def ask(question, temperature=0.7):
    prompt = f'<|user|>{question}<|end|><|assistant|>'
    ids = tokenizer.encode(prompt)
    ctx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(ctx, max_new_tokens=300, temperature=temperature, top_k=40, stop_tokens=stop)
    text = tokenizer.decode(out[0].tolist())
    response = text.split('<|assistant|>')[-1].replace('<|end|>', '').replace('<|user|>', '').strip()
    print(f'Q: {question}')
    print(f'A: {response}\n')

print('Model loaded! Try asking questions below.\n')

In [ ]:
# Ask questions! Change these or add your own.
ask('Ի՞նdelays է Արarat լdelays')
ask('Ov e grdelays Haykakan aybubendelays')
ask('Inchdelays e arevelyan hayerendelays')

In [ ]:
# Try your own question here!
ask('YOUR QUESTION HERE', temperature=0.7)

---
## What You Learned

| Stage | What happens | Result |
|-------|-------------|--------|
| **Stage 1** | Train on Wikipedia text | Model learns Armenian language patterns |
| **Stage 2** | Fine-tune on Q&A pairs | Model learns to answer questions |

This is exactly how real AI assistants (ChatGPT, Claude, etc.) are built!

### Things to try
- Change `PRESET` to `medium` for better results
- Change `temperature` when generating/asking
- Try Level 2 BPE tokenization: `!python data/prepare.py --tokenizer bpe`
- Change model size: `!python train.py --n_layer 8 --n_head 8 --n_embd 512`

### Metrics explained
| Metric | Meaning | Good value |
|--------|---------|------------|
| **Loss** | How wrong predictions are (lower = better) | < 2.0 |
| **Perplexity** | Characters the model is confused between | 4 - 8 |
| **Accuracy** | % of correct next-character predictions | 30 - 45% |